<a id="inicio"></a>

# Sentiment Analysis at Scale — Part 1: Training

## Objective

Train a binary sentiment-analysis model over the full Amazon Reviews dataset (~17M reviews) using PySpark's MLlib on AWS Glue, then serialize it to S3 for downstream inference.

## Methodology

- **Distributed data loading**: Read Parquet-formatted Amazon Reviews (`amazon-reviews-pds-parquet`) via Spark
- **Filtering and caching**: Restrict to the `Electronics` partition (~3.1M reviews) and repartition into S3 for stable access
- **Exploratory queries**: Star-rating distribution, top-reviewed products, temporal review trends
- **Text preprocessing**: Lowercasing, punctuation removal, null-body filtering
- **Feature engineering**: `Tokenizer` + `StopWordsRemover` + either TF-IDF (`HashingTF` + `IDF`) or `Word2Vec`
- **Binary classification**: Sentiment target defined as `star_rating >= 3`. Train via `LogisticRegression` or `DecisionTree`
- **Evaluation**: Area under the ROC curve on a 30% held-out test set
- **Serialization**: Persist the fitted pipeline to S3 via Spark's native `.save()` API

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. This notebook runs on AWS Glue (Spark) — a local re-run requires a Spark environment (PySpark 3.x) and AWS credentials with S3 access.
2. Set your AWS credentials in a `.env` file before launching Jupyter Lab (the Dockerfile + docker-compose.yml expect it).
3. Download the Amazon Reviews Parquet dataset (`amazon-reviews-pds-parquet`) to your working environment.
4. The training notebook writes a trained model to S3; the test notebook loads it back.

</div>

---

<a id="indice"></a>
## Contents

* [1. Introduction](#section1)
* [2. Exploratory Analysis](#section2)
* [3. Modeling](#section3)

---

Environment setup — enter your AWS Academy credentials in the `.env` file before launching Jupyter Lab so the Spark session can reach S3.

In [5]:
%iam_role arn:aws:iam::270830193283:role/LabRole 
%region us-east-1
%number_of_workers 2
%idle_timeout 30

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
It looks like there is a newer version of the kernel available. The latest version is 1.0.9 and you have 1.0.4 installed.
Please run `pip install --upgrade aws-glue-sessions` to upgrade your kernel
Current iam_role is None
iam_role has been set to arn:aws:iam::270830193283:role/LabRole.
Previous region: None
Setting new region to: us-east-1
Region is set to: us-east-1
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.


In [1]:
import os
s3_bucket = "s3://<SOURCE_BUCKET>"

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Session ID: 96c92309-2658-4960-85e5-17e20d7fdd79
Applying the following default arguments:
--glue_kernel_version 1.0.4
--enable-glue-datacatalog true
Waiting for session 96c92309-2658-4960-85e5-17e20d7fdd79 to get into ready status...
Session 96c92309-2658-4960-85e5-17e20d7fdd79 has been created.



In [11]:
!ls

capstone_test.ipynb  capstone_training.ipynb  data  img


Upload the dataset (available as `amazon-reviews-pds-parquet`) — this will be the working data for the rest of the notebook.

In [7]:
import os
!aws s3 cp --recursive ./data/amazon-reviews-pds-parquet s3://{SOURCE_BUCKET_PLACEHOLDER}/amazon-reviews-pds-parquet

[Output truncated — original upload transferred ~3000 Parquet files to S3.]


Additional configuration so Spark can persist models back to S3:

In [2]:
sc._jsc.hadoopConfiguration().set("mapred.output.committer.class", "org.apache.hadoop.mapred.DirectFileOutputCommitter")

---

<a id="section1"></a>
## 1. Introduction

This capstone trains a sentiment-detection model using Spark MLlib and AWS Glue.

The source dataset is **Amazon Reviews**, with the following schema:

```
marketplace       - 2-letter country code of the marketplace where the review was written
customer_id       - Random identifier that aggregates reviews from a single author
review_id         - Unique review ID
product_id        - Unique product ID (for multilingual datasets, the same product across regions shares this ID)
product_parent    - Random identifier that aggregates reviews for the same product
product_title     - Product name
product_category  - Broad category (also used as the partition key for the dataset)
star_rating       - 1-5 star rating
helpful_votes     - Number of helpful votes
total_votes       - Total votes the review received
vine              - Whether the review was part of the Vine program
verified_purchase - Whether the review was a verified purchase
review_headline   - Title of the review
review_body       - Review text
review_date       - Date the review was written
```

`product_category` is the partition key, which lets us work efficiently with a single category slice.

---

<a id="section2"></a>
## 2. Exploratory Analysis

Before modeling, we do a minimal exploration of the data to understand its shape and distributions.

---
### Task 1: Data Loading

Load the full Parquet dataset and count records. Don't persist yet.

In [10]:
import os
df = spark.read.parquet(f"s3://{os.environ['SOURCE_BUCKET']}/amazon-reviews-pds-parquet/")
df.count()

16967903


**Expected**: 16,967,903 records

---
### Task 2: Filtering

The full dataset is too large to train a model on. We restrict to the `Electronics` partition. Filter and count again. Don't cache yet.

In [11]:
import os
df_electronics = spark.read.parquet(f"s3://{os.environ['SOURCE_BUCKET']}/amazon-reviews-pds-parquet/product_category=Electronics/")
df_electronics.count()


3105119


**Expected**: 3,105,119 records

---
### Task 3: Storage

To avoid relying on the public bucket, we write the filtered data back to our own S3 bucket. Create a bucket for this capstone and write the data under the `electronics` prefix. Use `repartition(32)` to control file sizes. Reload from your own bucket and cache.

In [12]:
import os
output_path = f"s3://{os.environ['WORK_BUCKET']}/electronics/"

df_electronics.repartition(32).write.mode("overwrite").parquet(output_path)

In [14]:
df_electronics_new = spark.read.parquet(output_path)

df_electronics_new.cache()

df_electronics_new.count()




3105119


---
### Task 4: Queries

From the filtered dataset:

1. Show total review count per `star_rating` value
2. Top 10 products by `total_votes`, showing product title, vote count, and average `star_rating`
3. Review count and mean `star_rating` per (year, month), showing the last 15 records sorted by year and month

In [41]:
from pyspark.sql.functions import col

df_electronics_new.groupBy("star_rating").count().orderBy(col("star_rating")).show()


+-----------+-------+
|star_rating|  count|
+-----------+-------+
|          1| 359248|
|          2| 179834|
|          3| 239459|
|          4| 538824|
|          5|1787754|
+-----------+-------+


In [44]:
from pyspark.sql.functions import col, avg, sum as spark_sum

df_top10 = (
    df_electronics_new
    .groupBy("product_id", "product_title")
    .agg(
        spark_sum("total_votes").alias("total_votes"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("total_votes").desc())
    .limit(10)
)

df_top10.show(truncate=False)


+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|product_id|product_title                                                                                                                                                                                       |total_votes|avg_star_rating   |
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|B000I1X6PM|Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                                                                                                                     |46348      |3.4917491749174916|
|B000J36XR2|AudioQuest K2 Terminated

In [45]:
df_top10 = (
    df_electronics_new
    .groupBy("product_title")
    .agg(
        spark_sum("total_votes").alias("total_votes"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("total_votes").desc())
    .limit(10)
)

df_top10.show(truncate=False)


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|product_title                                                                                                                                                                                       |total_votes|avg_star_rating   |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                                                                                                                     |46348      |3.4917491749174916|
|Panasonic ErgoFit In-Ear Earbud Headphone                                      

In [47]:
from pyspark.sql.functions import year, month, col, avg, count


df_reviews_by_month = (
    df_electronics_new
    .withColumn("year", year(col("review_date")))
    .withColumn("month", month(col("review_date")))
    .groupBy("year", "month")
    .agg(
        count("*").alias("num_reviews"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("year").desc(), col("month").desc())
    .limit(15)
)

df_reviews_by_month.show(truncate=False)


+----+-----+-----------+------------------+
|year|month|num_reviews|avg_star_rating   |
+----+-----+-----------+------------------+
|2015|8    |102984     |4.093985473471608 |
|2015|7    |99806      |4.08580646454121  |
|2015|6    |91486      |4.093478783639027 |
|2015|5    |89357      |4.100439808856609 |
|2015|4    |93152      |4.102466935760907 |
|2015|3    |108861     |4.11561532596614  |
|2015|2    |107291     |4.118062092813004 |
|2015|1    |120404     |4.152602903558021 |
|2014|12   |107891     |4.120232456831432 |
|2014|11   |77529      |4.10810148460576  |
|2014|10   |78128      |4.114210014335449 |
|2014|9    |77753      |4.116111275449179 |
|2014|8    |82143      |4.115664146671049 |
|2014|7    |79424      |4.118352135374698 |
|2014|6    |48375      |4.0157726098191215|
+----+-----+-----------+------------------+


### Expected results

**1. Reviews per star rating**

| star_rating | count   |
|-------------|---------|
| 1           | 359,248  |
| 2           | 179,834  |
| 3           | 239,459  |
| 4           | 538,824  |
| 5           | 1,787,754 |

**2. Top 10 products by total votes** (selected rows)

| product_title                                                   | total_votes | star_rating |
|-----------------------------------------------------------------|-------------|-------------|
| Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer) | 12,944      | 3           |
| AudioQuest K2 Terminated Speaker Cable                          | 9,072       | 1           |
| Panasonic ErgoFit In-Ear Earbud Headphone                       | 8,680       | 5           |
| Apple iPod touch 8GB (4th Generation)                           | 6,353       | 5           |

**3. Review volume and mean rating per (year, month)** — most recent 15

| year | month | review_count | mean_star_rating |
|------|-------|--------------|------------------|
| 2015 | 8     | 102,984      | 4.094            |
| 2015 | 7     | 99,806       | 4.086            |
| 2015 | 6     | 91,486       | 4.093            |
| 2015 | 5     | 89,357       | 4.100            |
| 2015 | 4     | 93,152       | 4.102            |
| 2015 | 3     | 108,861      | 4.116            |
| 2015 | 2     | 107,291      | 4.118            |
| 2015 | 1     | 120,404      | 4.153            |
| 2014 | 12    | 107,891      | 4.120            |
| 2014 | 11    | 77,529       | 4.108            |
| 2014 | 10    | 78,128       | 4.114            |
| 2014 | 9     | 77,753       | 4.116            |
| 2014 | 8     | 82,143       | 4.116            |
| 2014 | 7     | 79,424       | 4.118            |
| 2014 | 6     | 48,375       | 4.016            |

---

<a id="section3"></a>
## 3. Modeling

Before modeling, we apply two passes of cleaning to the data.

---
### Task 5: Text preparation

Clean the `review_body` column using string operations and regexes:

- Lowercase the text
- Strip punctuation and digits
- Drop rows that end up with a null body after cleaning

Show results for the first 10 rows (sorted by `review_id`).

In [15]:
from pyspark.sql.functions import col, lower, regexp_replace, asc
df_limpio = (
    df_electronics
    .withColumn("review_body_clean", lower(col("review_body")))
    .withColumn("review_body_clean", regexp_replace(col("review_body_clean"), r"[^a-zA-Z\s]", ""))
    .filter(col("review_body_clean").isNotNull())
)


In [19]:
(
    df_limpio
    .orderBy(asc("review_id"))
    .select("review_id", "review_body", "review_body_clean")
    .limit(10)
    .toPandas()
)


        review_id  ...                                  review_body_clean
0  R10000WMGXS51T  ...  great little emergency radio  very good recept...
1  R10001L4QTCA84  ...  lives up to its claim and really does fit bulk...
2  R10003OLR2P5UE  ...  ive gone through three pairs of these in the l...
3  R10005O193PJ6W  ...  stopped working after a while changed batterie...
4  R10008LR7CU84N  ...  i ordered this cable and it doesnt work when i...
5  R10009JN2UWOJC  ...  have not owned it that long however it has the...
6  R1000AMVKPW32O  ...  bought for a gift and it is just what was need...
7  R1000CJMO2L8X4  ...                                perfect for the gym
8  R1000EDGJUU3CU  ...  love these  the sound quality is amazing  the ...
9  R1000EG9XXBLXT  ...  i have had good success with these disks and h...

[10 rows x 3 columns]


**Expected**: The cleaned column should strip all uppercase, punctuation, and digits from the raw review body.

---
### Task 6: Sentiment target

Create a binary `sentiment` variable: `1` if `star_rating >= 3`, else `0`. Use Spark's `when` function. Show the first 10 rows (sorted by `review_id`).

In [49]:
from pyspark.sql.functions import when, col, asc

df_sentiment = (
    df_limpio
    .withColumn(
        "sentiment",
        when(col("star_rating") < 3, 0).otherwise(1)
    )
    .orderBy(asc("review_id"))
)

df_sentiment_pd = (
    df_sentiment
    .select("review_id", "review_body", "star_rating", "sentiment")
    .limit(10)
    .toPandas()
)

df_sentiment_pd



        review_id  ...                                  review_body_clean
0  R10000WMGXS51T  ...  great little emergency radio  very good recept...
1  R10001L4QTCA84  ...  lives up to its claim and really does fit bulk...
2  R10003OLR2P5UE  ...  ive gone through three pairs of these in the l...
3  R10005O193PJ6W  ...  stopped working after a while changed batterie...
4  R10008LR7CU84N  ...  i ordered this cable and it doesnt work when i...
5  R10009JN2UWOJC  ...  have not owned it that long however it has the...
6  R1000AMVKPW32O  ...  bought for a gift and it is just what was need...
7  R1000CJMO2L8X4  ...                                perfect for the gym
8  R1000EDGJUU3CU  ...  love these  the sound quality is amazing  the ...
9  R1000EG9XXBLXT  ...  i have had good success with these disks and h...

[10 rows x 4 columns]


In [54]:
df_sentiment.select("review_id", "review_body", "star_rating", "sentiment").show(10)

+--------------+--------------------+-----------+---------+
|     review_id|         review_body|star_rating|sentiment|
+--------------+--------------------+-----------+---------+
|R10000WMGXS51T|Great little emer...|          5|        1|
|R10001L4QTCA84|Lives up to its c...|          5|        1|
|R10003OLR2P5UE|I've gone through...|          3|        1|
|R10005O193PJ6W|stopped working a...|          3|        1|
|R10008LR7CU84N|I ordered this ca...|          1|        0|
|R10009JN2UWOJC|Have not owned it...|          5|        1|
|R1000AMVKPW32O|Bought for a gift...|          5|        1|
|R1000CJMO2L8X4| Perfect for the gym|          5|        1|
|R1000EDGJUU3CU|Love these !!! Th...|          5|        1|
|R1000EG9XXBLXT|I have had good s...|          5|        1|
+--------------+--------------------+-----------+---------+
only showing top 10 rows


**Expected**: `sentiment` column populated; rows with `star_rating` in {3, 4, 5} get 1, rows with {1, 2} get 0.

---
### Task 7: Train/Test Split

Split into train (70%) and test (30%). Save the test set to the S3 bucket under the `electronics_test` prefix.

In [55]:
train_df, test_df = df_sentiment.randomSplit([0.7, 0.3], seed=42)


In [56]:
import os
test_df.write.mode("overwrite").parquet(f"s3://{os.environ['WORK_BUCKET']}/electronics_test/")


To train a sentiment classifier we first need to vectorize the text. Spark MLlib ships two standard approaches — **TF-IDF** and **Word2Vec** — both of which convert text strings into numeric feature vectors suitable for a downstream classifier.

---
### Task 8: Sentiment Pipeline

Build a Spark ML `Pipeline` combining transformers and estimators:

- **`Tokenizer`** — convert sentences into lists of tokens
- **`StopWordsRemover`** — drop low-information words
- **Feature extraction** — choose one:
    - TF-IDF using `HashingTF` + `IDF`
    - `Word2Vec`
- **Classifier** — `LogisticRegression` or `DecisionTree` (avoid ensembles due to training time)

Wire them up into a pipeline. A few tips:

- Use `.sample()` to downsample during development — full-dataset training is slow
- Hyperparameter tuning is possible but can be very slow
- **Evaluate on the test set using AUC (area under ROC curve)**

In [57]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [58]:
train_sample = train_df.sample(withReplacement=False, fraction=0.05, seed=42)
test_sample = test_df.sample(withReplacement=False, fraction=0.05, seed=42)



In [59]:
tokenizer = Tokenizer(inputCol="review_body_clean", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
word2vec = Word2Vec(inputCol="filtered_words", outputCol="features", vectorSize=100, minCount=5)
lr = LogisticRegression(featuresCol="features", labelCol="sentiment")

pipeline = Pipeline(stages=[tokenizer, remover, word2vec, lr])


In [60]:
model = pipeline.fit(train_sample)

---
### Task 9: Model Serialization

Save the trained model to S3 (into the bucket you created earlier) using Spark's native model persistence API.

In [62]:
import os
model.save(f"s3://{os.environ['WORK_BUCKET']}/word2vec_model/")